In [ ]:
!pip install -q transformers datasets accelerate torch

In [1]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)

set_seed(42)  # set seed for reproducibility

print('GPU available:', torch.cuda.is_available())

GPU available: True


In [2]:
import json

# Load training data
with open('train.json', 'r', encoding='utf-8') as f:
    train_examples = json.load(f)

# Load validation data
with open('validation.json', 'r', encoding='utf-8') as f:
    validation_examples = json.load(f)

# Check token and tag counts
for example in train_examples + validation_examples:
    assert len(example['tokens']) == len(example['tags'])

print('Training examples:', len(train_examples))
print('Validation examples:', len(validation_examples))

Training examples: 166
Validation examples: 56


In [3]:
# @title create label-to-id and id-to-label mapping

label_list = [
    'O',
    'B-CHEM_ENT',
    'I-CHEM_ENT',
    'B-MAT_ENT_STRUCT',
    'I-MAT_ENT_STRUCT',
    'B-MAT_ENT_UNSTRUCT',
    'I-MAT_ENT_UNSTRUCT',
    'B-PROPERTY',
    'I-PROPERTY',
    'B-END_USE',
    'I-END_USE',
    'B-PROCESS',
    'I-PROCESS',
    'B-EQUIPMENT',
    'I-EQUIPMENT',
    'B-MEASUREMENT',
    'I-MEASUREMENT',
    'B-ABBREVIATION',
    'I-ABBREVIATION',
]

# map label names to numeric ids
label2id = {}

for index, label in enumerate(label_list):
    label2id[label] = index

# map numeric ids back to label names
id2label = {}

for label, index in label2id.items():
    id2label[index] = label

print(label2id)
print(id2label)


{'O': 0, 'B-CHEM_ENT': 1, 'I-CHEM_ENT': 2, 'B-MAT_ENT_STRUCT': 3, 'I-MAT_ENT_STRUCT': 4, 'B-MAT_ENT_UNSTRUCT': 5, 'I-MAT_ENT_UNSTRUCT': 6, 'B-PROPERTY': 7, 'I-PROPERTY': 8, 'B-END_USE': 9, 'I-END_USE': 10, 'B-PROCESS': 11, 'I-PROCESS': 12, 'B-EQUIPMENT': 13, 'I-EQUIPMENT': 14, 'B-MEASUREMENT': 15, 'I-MEASUREMENT': 16, 'B-ABBREVIATION': 17, 'I-ABBREVIATION': 18}
{0: 'O', 1: 'B-CHEM_ENT', 2: 'I-CHEM_ENT', 3: 'B-MAT_ENT_STRUCT', 4: 'I-MAT_ENT_STRUCT', 5: 'B-MAT_ENT_UNSTRUCT', 6: 'I-MAT_ENT_UNSTRUCT', 7: 'B-PROPERTY', 8: 'I-PROPERTY', 9: 'B-END_USE', 10: 'I-END_USE', 11: 'B-PROCESS', 12: 'I-PROCESS', 13: 'B-EQUIPMENT', 14: 'I-EQUIPMENT', 15: 'B-MEASUREMENT', 16: 'I-MEASUREMENT', 17: 'B-ABBREVIATION', 18: 'I-ABBREVIATION'}


In [4]:
# define a function to convert tag names to numeric ids
# and create a hugging face dataset
def make_dataset(examples):
    data = []

    for example in examples:
        tag_ids = []

        # replace each tag name with its numeric id
        for tag in example['tags']:
            tag_id = label2id[tag]
            tag_ids.append(tag_id)

        item = {
            'tokens': example['tokens'],
            'ner_tags': tag_ids,
        }

        data.append(item)

    # convert the list of dictionaries into a hugging face dataset
    dataset = Dataset.from_list(data)

    return dataset

In [5]:
# create hugging face datasets for training and validation
train_dataset = make_dataset(train_examples)
validation_dataset = make_dataset(validation_examples)

# print the dataset summary
print(train_dataset)

# print the actual data in the first row
print(train_dataset[0])

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 166
})
{'tokens': ['The', 'cryo', '-', 'fractured', 'surface', 'of', 'PP', 'samples', 'was', 'coated', 'with', 'gold', 'vapor', 'and', 'observed', 'under', 'SEM', '(', 'JSM', '-', '7400F', ',', 'JEOL', ',', 'Japan', ')', 'to', 'evaluate', 'the', 'distribution', 'of', 'CNC', 'in', 'the', 'PP', 'matrix', '.'], 'ner_tags': [0, 11, 12, 12, 0, 0, 5, 6, 0, 11, 0, 3, 4, 0, 0, 0, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 6, 0]}


In [6]:
# load the tokenizer associated with this BERT checkpoint
checkpoint = 'google-bert/bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    use_fast=True,
)

In [7]:
def split_into_chunks(words, labels):
    max_tokens = 512 - tokenizer.num_special_tokens_to_add(pair=False)

    # Count how many subtokens belong to each word.
    encoded = tokenizer(
        words,
        is_split_into_words=True,
        add_special_tokens=False,
        truncation=False,
    )

    token_counts = [0] * len(words)

    for word_id in encoded.word_ids():
        if word_id is not None:
            token_counts[word_id] += 1

    chunks = []
    start = 0

    while start < len(words):
        end = start
        token_count = 0

        while end < len(words):
            if token_count + token_counts[end] > max_tokens:
                break

            token_count += token_counts[end]
            end += 1

        # Move the boundary back if it would split an entity.
        if end < len(words):
            next_label = id2label[labels[end]]

            while end > start and next_label.startswith("I-"):
                end -= 1
                next_label = id2label[labels[end]]

        if end == start:
            raise ValueError(
                "A word or entity exceeds BERT's token limit. "
                "Inspect this example before continuing."
            )

        chunks.append((words[start:end], labels[start:end]))
        start = end

    return chunks


def tokenize_and_align_labels(examples):
    chunk_words = []
    chunk_labels = []

    for words, labels in zip(examples["tokens"], examples["ner_tags"]):
        for words_part, labels_part in split_into_chunks(words, labels):
            chunk_words.append(words_part)
            chunk_labels.append(labels_part)

    tokenized = tokenizer(
        chunk_words,
        is_split_into_words=True,
        truncation=False,
    )

    aligned_labels = []

    for index, labels in enumerate(chunk_labels):
        word_ids = tokenized.word_ids(batch_index=index)
        previous_word_id = None
        token_labels = []

        for word_id in word_ids:
            if word_id is None or word_id == previous_word_id:
                token_labels.append(-100)
            else:
                token_labels.append(labels[word_id])

            previous_word_id = word_id

        aligned_labels.append(token_labels)

    tokenized["labels"] = aligned_labels
    return tokenized

In [8]:
tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names,
    load_from_cache_file=False,
)

tokenized_validation = validation_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=validation_dataset.column_names,
    load_from_cache_file=False,
)

for name, original, tokenized in [
    ("Train", train_dataset, tokenized_train),
    ("Validation", validation_dataset, tokenized_validation),
]:
    original_words = sum(len(row["tokens"]) for row in original)

    labeled_words = sum(
        sum(label != -100 for label in row["labels"])
        for row in tokenized
    )

    longest_input = max(len(row["input_ids"]) for row in tokenized)

    assert original_words == labeled_words, f"{name}: some words were lost"
    assert longest_input <= 512, f"{name}: input exceeds BERT's limit"

    print(f"{name}: {len(original)} paragraphs → {len(tokenized)} chunks")
    print(f"Words retained: {labeled_words}/{original_words}")
    print(f"Longest input: {longest_input} tokens\n")

Map:   0%|          | 0/166 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (650 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Train: 166 paragraphs → 184 chunks
Words retained: 32827/32827
Longest input: 512 tokens

Validation: 56 paragraphs → 58 chunks
Words retained: 9539/9539
Longest input: 512 tokens



In [9]:
# inspect the BERT tokens and their aligned labels for one sentence
sample = tokenized_train[0]

tokens = tokenizer.convert_ids_to_tokens(sample['input_ids'])

for token, label_id in zip(tokens, sample['labels']):

    if label_id == -100:
        label = 'IGNORED'
    else:
        label = id2label[label_id]

    print(token, '->', label)

[CLS] -> IGNORED
The -> O
cry -> B-PROCESS
##o -> IGNORED
- -> I-PROCESS
fractured -> I-PROCESS
surface -> O
of -> O
PP -> B-MAT_ENT_UNSTRUCT
samples -> I-MAT_ENT_UNSTRUCT
was -> O
coated -> B-PROCESS
with -> O
gold -> B-MAT_ENT_STRUCT
vapor -> I-MAT_ENT_STRUCT
and -> O
observed -> O
under -> O
SE -> B-EQUIPMENT
##M -> IGNORED
( -> O
J -> O
##SM -> IGNORED
- -> O
74 -> O
##00 -> IGNORED
##F -> IGNORED
, -> O
J -> O
##E -> IGNORED
##OL -> IGNORED
, -> O
Japan -> O
) -> O
to -> O
evaluate -> O
the -> O
distribution -> O
of -> O
C -> B-MAT_ENT_UNSTRUCT
##NC -> IGNORED
in -> O
the -> O
PP -> B-MAT_ENT_UNSTRUCT
matrix -> I-MAT_ENT_UNSTRUCT
. -> O
[SEP] -> IGNORED


In [10]:
# load pretrained BERT with a token classification layer for NER
# the classification layer starts with newly initialized weights
model = AutoModelForTokenClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params 

In [11]:
# pad sentences to the longest sequence in each training batch
# padding positions receive attention_mask=0 and labels=-100
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
)

In [12]:
training_args = TrainingArguments(
    output_dir='./models',
    learning_rate=2e-5,             # Size of weight updates.
    per_device_train_batch_size=2,  # Training sentences per batch per device.
    per_device_eval_batch_size=2,   # Validation sentences per batch per device.
    num_train_epochs=10,             # Ten passes through this small training set.
    eval_strategy='epoch',          # Evaluate after each epoch.
    save_strategy='no',             # Save manually after training.
    logging_steps=10,
    report_to='none',               # Disable external logging services.
    seed=42,
)


In [13]:
# Connect the model, datasets, tokenizer, and training settings.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
)


In [14]:
print(checkpoint)

google-bert/bert-base-cased


In [15]:
# Start fine-tuning: update BERT and the NER classification layer.
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.703721,0.787138
2,0.586168,0.583455
3,0.521123,0.513947
4,0.379665,0.487361
5,0.331770,0.482102
6,0.264534,0.504192
7,0.235562,0.525197
8,0.200351,0.532427
9,0.195439,0.543603
10,0.145314,0.547959


TrainOutput(global_step=920, training_loss=0.39976898470650546, metrics={'train_runtime': 72.3352, 'train_samples_per_second': 25.437, 'train_steps_per_second': 12.719, 'total_flos': 286051630899720.0, 'train_loss': 0.39976898470650546, 'epoch': 10.0})

In [16]:
!pip install seqeval -q

In [17]:
import numpy as np
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

predictions, labels, metrics = trainer.predict(tokenized_validation)

predicted_ids = np.argmax(predictions, axis=2)

true_predictions = []
true_labels = []

for prediction, label in zip(predicted_ids, labels):

    predicted_tags = []
    actual_tags = []

    for pred_id, label_id in zip(prediction, label):

        if label_id != -100:
            predicted_tags.append(label_list[pred_id])
            actual_tags.append(label_list[label_id])

    true_predictions.append(predicted_tags)
    true_labels.append(actual_tags)

print('Precision:', precision_score(true_labels, true_predictions))
print('Recall:', recall_score(true_labels, true_predictions))
print('F1-score:', f1_score(true_labels, true_predictions))

print('\nPer-label performance:')
print(classification_report(true_labels, true_predictions, digits=4))

Precision: 0.4979050279329609
Recall: 0.6032148900169205
F1-score: 0.5455241009946442

Per-label performance:
                  precision    recall  f1-score   support

    ABBREVIATION     0.4086    0.6230    0.4935        61
        CHEM_ENT     0.4777    0.6410    0.5474       117
         END_USE     0.6667    0.6667    0.6667        18
       EQUIPMENT     0.4655    0.4355    0.4500        62
  MAT_ENT_STRUCT     0.0000    0.0000    0.0000        33
MAT_ENT_UNSTRUCT     0.5298    0.6434    0.5811       387
     MEASUREMENT     0.6836    0.7756    0.7267       156
         PROCESS     0.4095    0.5513    0.4699       156
        PROPERTY     0.4303    0.5469    0.4817       192

       micro avg     0.4979    0.6032    0.5455      1182
       macro avg     0.4524    0.5426    0.4908      1182
    weighted avg     0.4906    0.6032    0.5398      1182



In [ ]:
# Create an inference pipeline using the fine-tuned model.
# Use the first subtoken's prediction to choose each word's label.
ner = pipeline(
    task='token-classification',
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy='first',
    device=0 if torch.cuda.is_available() else -1,
)

In [ ]:
text = 'bacterial cellulose was mixed with ethanol.'

# Predict entities in the new sentence.
# This tiny training dataset may produce incorrect predictions.
results = ner(text)

for entity in results:
    print({
        'text': text[entity['start']:entity['end']],
        'label': entity['entity_group'],
        'score': round(entity['score'], 3),
    })

{'text': 'bacterial cellulose', 'label': 'MATERIAL', 'score': np.float32(0.932)}
{'text': 'ethanol', 'label': 'CHEMICAL', 'score': np.float32(0.969)}


In [ ]:
# Save both the fine-tuned model and its tokenizer.
# Files saved here are temporary unless downloaded or copied elsewhere.
trainer.save_model('./my_bert_ner')
tokenizer.save_pretrained('./my_bert_ner')

{'text': 'bacterial cellulose', 'label': 'MATERIAL', 'score': np.float32(0.803)}
{'text': 'ethanol', 'label': 'CHEMICAL', 'score': np.float32(0.962)}


In [ ]:
# Reload the saved model and tokenizer without training again.
# Use CPU to avoid loading another copy onto the GPU.
reloaded_ner = pipeline(
    task='token-classification',
    model='./my_bert_ner',
    tokenizer='./my_bert_ner',
    aggregation_strategy='first',
    device=-1,
)

results = reloaded_ner(text)

for entity in results:
    print({
        'text': text[entity['start']:entity['end']],
        'label': entity['entity_group'],
        'score': round(entity['score'], 3),
    })

In [ ]:
# Optional: download the saved model and tokenizer from Colab as a ZIP file.
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    'my_bert_ner',
    'zip',
    root_dir='./my_bert_ner',
)

files.download(archive_path)